# Exploring Subjecthood in `multilingual BERT`

Code for the fine-tuning, evaluating and analyzing `multilingual BERT` from the graduation thesis 'Exploring Subjecthood in Multilingual Language Models'.

## Installations & Imports

In this Section, all needed modules and libraries are downloaded and imported.

In order to run code and save fine-tuned models, the token from [Hugging Face account](https://huggingface.co/) is needed.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%pip install -U kaleido

In [ ]:
%pip install plotly==6.1.1 pyconll transformers datasets evaluate seqeval huggingface_hub scikit-learn

In [ ]:
!apt update
!apt install -y chromium-browser chromium-chromedriver \
libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 \
libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2

In [ ]:
import kaleido
import os
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

In [ ]:
import pyconll
import statistics
from pprint import pprint
import numpy as np
from collections import Counter
import tqdm
import json
import random

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torch.utils.data import DataLoader

import evaluate
from sklearn.metrics import accuracy_score, classification_report, hamming_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from huggingface_hub import notebook_login

In [ ]:
kaleido.get_chrome_sync()

In [ ]:
os.environ["KALIEDO_PATH"] = "/usr/bin/chromium-browser"
os.environ["CHROMIUM_PATH"] = "/usr/bin/chromium-browser"

In [ ]:
notebook_login()

## Data Download

In this Section, the data from [Universal Dependencies](https://universaldependencies.org/) is downloaded.

In [ ]:
%mkdir data
%cd data

In [ ]:
data = {
    "Arabic": {
        "treebank": "UD_Arabic-PADT",
        "path": "/content/data/Arabic/ar_padt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Arabic-PADT/refs/heads/master/ar_padt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-arabic",
    },
    "Basque": {
        "treebank": "UD_Basque-BDT",
        "path": "/content/data/Basque/eu_bdt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Basque-BDT/refs/heads/master/eu_bdt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-basque",
    },
    "English": {
        "treebank": "UD_English-EWT",
        "path": "/content/data/English/en_ewt-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-english",
    },
    "German": {
        "treebank": "UD_German-GSD",
        "path": "/content/data/German/de_gsd-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_German-GSD/refs/heads/master/de_gsd-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-german",
    },
    "Hindi": {
        "treebank": "UD_Hindi-HDTB",
        "path": "/content/data/Hindi/hi_hdtb-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/refs/heads/master/hi_hdtb-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-hindi",
    },
    "Russian": {
        "treebank": "UD_Russian-GSD",
        "path": "/content/data/Russian/ru_gsd-ud",
        "train": "https://github.com/UniversalDependencies/UD_Russian-GSD/raw/refs/heads/master/ru_gsd-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-GSD/refs/heads/master/ru_gsd-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-GSD/refs/heads/master/ru_gsd-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-russian",
    },
    "Turkish": {
        "treebank": "UD_Turkish-Penn",
        "path": "/content/data/Turkish/tr_penn-ud",
        "train": "https://github.com/UniversalDependencies/UD_Turkish-Penn/raw/refs/heads/master/tr_penn-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Penn/refs/heads/master/tr_penn-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-Penn/refs/heads/master/tr_penn-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-turkish",
    },
    "Welsh": {
        "treebank": "UD_Welsh-CCG",
        "path": "/content/data/Welsh/cy_ccg-ud",
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-train.conllu",
        "dev": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-dev.conllu",
        "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_Welsh-CCG/refs/heads/master/cy_ccg-ud-test.conllu",
        "model": "fromdeath2morning/mbert-argumentClassification-welsh",
    },
}

In [ ]:
for language in data:
    %mkdir "{language}"
    %cd "{language}"
    for mode in ["train", "dev", "test"]:
        !wget "{data[language][mode]}"
    %cd ..

## Experiments

In this Section, I conduct the experiments with `multilingual BERT` in order to explore the subjecthood representations.

### Fine-tuning and evaluating `mBERT`. Training and evaluating logreg probes

This subsection is dedicated to fine-tuning `mBERT` with the following task:
*   token classification;
*   each token has one of four labels: `None`, `S`, `A` or `P` (core arguments or not core arguments).

Fine-tuned models are pushed to the Hugging Face.



#### Experimental setup

Here, I provide the functions that are essential for fine-tuning and evaluating `mBERT` and training classifiers on its hidden states.

In [ ]:
def classifyTokens(sentence, filter=[], filterType=None):
    """
    This function defines the label (None, S, A or P) to each token of the sentence based on the conllu tree structure and deprel.
    :param sentence: sentence from the conllu file
    :param filter: list of the strings that represent the word order, used for filtration of the data
    :filterType: `include` in or `exclude` from the data the word orders from filter
    :returns: list of tokens' labels
    """

    labels = {}

    targetPOS = ["PRON", "NOUN", "PROPN"]

    for token in sentence:
        if token.id.isdigit():
            labels[int(token.id)] = "None"

    for token in sentence:
        if token.id.isdigit():

            if token.upos == "VERB":
                head = token

                childrenS = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPOS
                    and "nsubj" in token.deprel
                ]

                childrenP = [
                    int(token.id)
                    for token in sentence
                    if token.head == head.id
                    and token.upos in targetPOS
                    and "obj" in token.deprel
                ]

                if len(filter) == 0:
                    if len(childrenP) > 0:
                        for id_ in childrenS:
                            labels[int(id_)] = "A"

                        for id_ in childrenP:
                            labels[int(id_)] = "P"
                    else:
                        for id_ in childrenS:
                            labels[int(id_)] = "S"
                else:
                    order = []
                    if len(childrenP) > 0:
                        order.append((max(childrenP), "O"))

                    if len(childrenS) > 0:
                        order.append((max(childrenS), "S"))

                    order.append((int(head.id), "V"))

                    order = sorted(order, key=lambda x: x[0])

                    orderString = "".join([order[i][1] for i in range(len(order))])

                    if filterType == "exclude":
                        if orderString not in filter:
                            if len(childrenP) > 0:
                                for id_ in childrenS:
                                    labels[int(id_)] = "A"

                                for id_ in childrenP:
                                    labels[int(id_)] = "P"
                            else:
                                for id_ in childrenS:
                                    labels[int(id_)] = "S"
                        else:
                            return []
                    elif filterType == "include":
                        if orderString in filter:
                            if len(childrenP) > 0:
                                for id_ in childrenS:
                                    labels[int(id_)] = "A"

                                for id_ in childrenP:
                                    labels[int(id_)] = "P"
                            else:
                                for id_ in childrenS:
                                    labels[int(id_)] = "S"
                        else:
                            return []

    if len(filter) > 0 and filterType == "include":
        if set(labels.values()) == set(["None"]):
            return []

    labels2id = {"None": 0, "S": 1, "A": 2, "P": 3}
    result = []

    for label in sorted(labels):
        result.append(labels2id[labels[label]])

    return result

In [ ]:
def preprocessData(path, threshold=None, filter=[], filterType=None):
    """
    This function preprocessed conllu data in order to create later a transformer's dataset.
    :param path: the path to the conllu data
    :param threshold: the upper bound for the number of tokens in data, used for reducing the dataset
    :param filter: list of the strings that represent the word order, used for filtration of the data
    :filterType: `include` in or `exclude` from the data the word orders from filter
    :returns: a dict with preprocessed data: id, texts of the sentences, list of tokens and labels
    """
    random.seed(42)

    text = pyconll.load_from_file(path)
    data = {
        "id": [],
        "text": [],
        "tokens": [],
        "label": [],
    }

    if threshold is not None:
        random.shuffle(text)

    cnt = 0

    for sentence in text:
        tokenClasses = classifyTokens(sentence, filter=filter, filterType=filterType)

        if len(tokenClasses) > 0:
            data["id"].append(sentence.id)
            data["text"].append(sentence.text)
            data["tokens"].append([word.form for word in sentence if word.id.isdigit()])
            data["label"].append(tokenClasses)

            if threshold is not None:
                cnt += len(tokenClasses)
                if cnt > threshold:
                    return data

    return data

In [ ]:
def tokenize(batch):
    """
    This function tokenizes the batches from conllu data.
    :param batch: batch of data
    :returns: tokenized batch of data
    """

    tokenized_inputs = tokenizer(
        batch["tokens"], padding="max_length", max_length=512, truncation=True, is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(batch["label"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
def computeMetrics(output):
    """
    This function calculates the accuracy, weighted accuracy, hamming loss and classification report from the model's output.
    :param output: models output
    :returns: a dict with calculated metrics
    """
    logits, labels = output
    predictions = np.argmax(logits, axis=-1)

    label_list = ["None", "S", "A", "P"]

    true_predictions = []
    true_labels = []
    weights = []

    for prediction, label in zip(predictions, labels):
        true_labels += [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        true_predictions += [
            label_list[p] for (p, l) in zip(prediction, label) if l != -100
        ]
        weights += [
            1 if label_list[l] != "None" else 0
            for (p, l) in zip(prediction, label)
            if l != -100
        ]

    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "w_accuracy": accuracy_score(
            true_labels, true_predictions, sample_weight=weights
        ),
        "classification_report": classification_report(
            true_labels, true_predictions, output_dict=True, target_names=label_list
        ),
    }

In [ ]:
def train(
    model, epochs, trainPath, modelId, threshold=None, filter=[], filterType=None
):
    """
    This function trains the model on the token classification of the core arguments.
    :param model: the name of the model
    :param epochs: the number of training epochs
    :param trainPath: the path to the train data
    :param hub_model_id: the name of the fine-tuned model in order to push it to the hugging face hub
    :param threshold: the upper bound for the number of tokens in data, used for reducing the dataset
    :param filter: list of the strings that represent the word order, used for filtration of the data
    :filterType: `include` in or `exclude` from the data the word orders from filter
    :returns: 0 after successful execution
    """

    trainDataset = Dataset.from_dict(
        preprocessData(
            trainPath, threshold=threshold, filter=filter, filterType=filterType
        )
    )

    trainTokenized = trainDataset.map(
        tokenize,
        batched=True,
        batch_size=None,
        remove_columns=trainDataset.column_names,
    )

    model = AutoModelForTokenClassification.from_pretrained(
        model, num_labels=4, device_map="auto"
    )

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding=True,
    )

    trainingArgs = TrainingArguments(
        num_train_epochs=epochs,
        logging_strategy="steps",
        logging_steps=500,
        save_strategy="epoch",
        hub_model_id=modelId,
        push_to_hub=True,
        report_to="none",
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=trainingArgs,
        processing_class=tokenizer,
        data_collator=data_collator,
        train_dataset=trainTokenized,
    )

    trainer.train()
    trainer.push_to_hub(commit_message=f"Uploaded trained model: {modelId}")

    return 0

In [ ]:
def evaluate(modelId, testPath, threshold=None, filter=[], filterType=None):
    """
    This function evaluates the model on the token classification of the core arguments.
    :param modelId: the name of the model
    :param testPath: the path to the test data
    :param threshold: the upper bound for the number of tokens in data, used for reducing the dataset
    :param filter: list of the strings that represent the word order, used for filtration of the data
    :filterType: `include` in or `exclude` from the data the word orders from filter
    :returns: dict with computed metrics
    """

    testDataset = Dataset.from_dict(
        preprocessData(
            testPath, threshold=threshold, filter=filter, filterType=filterType
        )
    )

    testTokenized = testDataset.map(
        tokenize, batched=True, batch_size=None, remove_columns=testDataset.column_names
    )

    model = AutoModelForTokenClassification.from_pretrained(
        modelId, num_labels=4, device_map="cuda"
    )

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding=True,
    )

    evaluatingArgs = TrainingArguments(
        do_train=False,
        do_eval=True,
        hub_model_id=modelId,
        push_to_hub=True,
        report_to="none",
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=evaluatingArgs,
        processing_class=tokenizer,
        data_collator=data_collator,
        eval_dataset=testTokenized,
        compute_metrics=computeMetrics,
    )

    metrics = trainer.evaluate()
    # trainer.push_to_hub(commit_message=f"Computed metrics for {modelId}")

    return metrics

In [ ]:
def trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=None, thresholdTest=None
):
    """
    This function trains and tests Logistic regression on hidden states of the model and calculates classification report per layer.
    :param modelId: the name of the model
    :param trainPath: the path to the train data
    :param testPath: the path to the test data
    :param thresholdTrain: the upper bound for the number of tokens in train data, used for reducing the train dataset
    :param thresholdTest: the upper bound for the number of tokens in test data, used for reducing the test dataset
    :returns: dict with computed metrics
    """

    model = AutoModelForTokenClassification.from_pretrained(
        modelId,
        num_labels=4,
        device_map="cuda",
        output_hidden_states=True,
    )
    model.eval()

    label_list = ["None", "S", "A", "P"]

    trainDataset = Dataset.from_dict(
        preprocessData(trainPath, threshold=thresholdTrain)
    )

    trainTokenized = trainDataset.map(
        tokenize,
        batched=True,
        batch_size=None,
        remove_columns=trainDataset.column_names,
    )
    trainTokenized.set_format(
        type="torch", columns=["input_ids", "attention_mask", "labels"]
    )
    trainDataLoader = DataLoader(trainTokenized, batch_size=8, shuffle=True)

    testDataset = Dataset.from_dict(preprocessData(testPath, threshold=thresholdTest))
    testTokenized = testDataset.map(
        tokenize, batched=True, batch_size=None, remove_columns=testDataset.column_names
    )
    testTokenized.set_format(
        type="torch", columns=["input_ids", "attention_mask", "labels"]
    )
    testDataLoader = DataLoader(testTokenized, batch_size=8)

    metrics = {}

    for i in tqdm.tqdm(range(13)):
        print(f"Training LogReg classifier on layer {i}")

        trainHiddenStates, trainLabelList = [], []
        with torch.no_grad():
            for batch in trainDataLoader:
                batch = {k: v.to("cuda") for k, v in batch.items()}

                input_ids = batch["input_ids"]
                attention_mask = batch["attention_mask"]

                output = model(input_ids=input_ids, attention_mask=attention_mask)

                trainHiddenStates.append(output.hidden_states[i].cpu())
                trainLabelList.append(batch["labels"].cpu())

            trainHiddenStates = torch.cat(trainHiddenStates, dim=0)
            trainLabelList = torch.cat(trainLabelList, dim=0)

        logreg = LogisticRegression(max_iter=1000, random_state=42)

        X_train = trainHiddenStates.reshape(-1, trainHiddenStates.size(-1)).numpy()
        y_train = trainLabelList.reshape(-1).numpy()

        mask = y_train != -100
        X_train = X_train[mask]
        y_train = y_train[mask]

        logreg.fit(X_train, y_train)

        print(f"Testing LogReg classifier on layer {i}")

        testHiddenStates, testLabelList = [], []

        with torch.no_grad():
            for batch in testDataLoader:
                batch = {k: v.to("cuda") for k, v in batch.items()}

                input_ids = batch["input_ids"]
                attention_mask = batch["attention_mask"]

                output = model(input_ids=input_ids, attention_mask=attention_mask)

                testHiddenStates.append(output.hidden_states[i].cpu())
                testLabelList.append(batch["labels"].cpu())

            testHiddenStates = torch.cat(testHiddenStates, dim=0)
            testLabelList = torch.cat(testLabelList, dim=0)

        X_test = testHiddenStates.reshape(-1, testHiddenStates.size(-1)).numpy()
        y_test = testLabelList.reshape(-1).numpy()

        mask = y_test != -100
        X_test = X_test[mask]
        y_test = y_test[mask]

        y_pred = logreg.predict(X_test)

        metrics[f"layer {i}"] = classification_report(
            y_test, y_pred, output_dict=True, target_names=label_list
        )

    return metrics

#### Results

In this subsection, I fine-tune and evaluate `mBERT` and train classifiers on its hidden states.

**Arabic**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Arabic/ar_padt-ud-train.conllu"
modelId = "mbert-argumentClassification-arabic"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Arabic/ar_padt-ud-test.conllu"
modelId = data["Arabic"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers



In [ ]:
modelId = data["Arabic"]["model"]

trainPath = "/content/data/Arabic/ar_padt-ud-train.conllu"
testPath = "/content/data/Arabic/ar_padt-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Arabic_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**Basque**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Basque/eu_bdt-ud-train.conllu"
modelId = "mbert-argumentClassification-basque"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Basque/eu_bdt-ud-test.conllu"
modelId = data["Basque"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["Basque"]["model"]

trainPath = "/content/data/Basque/eu_bdt-ud-train.conllu"
testPath = "/content/data/Basque/eu_bdt-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Basque_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**English**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/English/en_ewt-ud-train.conllu"
modelId = "mbert-argumentClassification-english"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/English/en_ewt-ud-test.conllu"
modelId = data["English"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["English"]["model"]

trainPath = "/content/data/English/en_ewt-ud-train.conllu"
testPath = "/content/data/English/en_ewt-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_English_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**German**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/German/de_gsd-ud-train.conllu"
modelId = "mbert-argumentClassification-german"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/German/de_gsd-ud-test.conllu"
modelId = data["German"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["German"]["model"]

trainPath = "/content/data/German/de_gsd-ud-train.conllu"
testPath = "/content/data/German/de_gsd-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_German_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**Hindi**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Hindi/hi_hdtb-ud-train.conllu"
modelId = "mbert-argumentClassification-hindi"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Hindi/hi_hdtb-ud-test.conllu"
modelId = data["Hindi"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["Hindi"]["model"]

trainPath = "/content/data/Hindi/hi_hdtb-ud-train.conllu"
testPath = "/content/data/Hindi/hi_hdtb-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Hindi_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**Russian**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Russian/ru_gsd-ud-train.conllu"
modelId = "mbert-argumentClassification-russian"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Russian/ru_gsd-ud-test.conllu"
modelId = data["Russian"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["Russian"]["model"]

trainPath = "/content/data/Russian/ru_gsd-ud-train.conllu"
testPath = "/content/data/Russian/ru_gsd-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Russian_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**Turkish**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Turkish/tr_penn-ud-train.conllu"
modelId = "mbert-argumentClassification-turkish"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Turkish/tr_penn-ud-test.conllu"
modelId = data["Turkish"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["Turkish"]["model"]

trainPath = "/content/data/Turkish/tr_penn-ud-train.conllu"
testPath = "/content/data/Turkish/tr_penn-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Turkish_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

**Welsh**

I. Training

In [ ]:
epochs = 10
model = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

trainPath = "/content/data/Welsh/cy_ccg-ud-train.conllu"
modelId = "mbert-argumentClassification-welsh"

train(model, epochs, trainPath, modelId)

II. Evaluating

In [ ]:
testPath = "/content/data/Welsh/cy_ccg-ud-test.conllu"
modelId = data["Welsh"]["model"]

tokenizer = AutoTokenizer.from_pretrained(modelId)

metrics = evaluate(modelId, testPath)

III. Training classifiers

In [ ]:
modelId = data["Welsh"]["model"]

trainPath = "/content/data/Welsh/cy_ccg-ud-train.conllu"
testPath = "/content/data/Welsh/cy_ccg-ud-test.conllu"

tokenizer = AutoTokenizer.from_pretrained(modelId)

results = trainClassifiers(
    modelId, trainPath, testPath, thresholdTrain=20000, thresholdTest=8500
)

with open(f"metrics_Welsh_per_layer.json", "w") as file:
    json.dump(results, file, indent=4)

#### Analysis of layers

In this subsection, I analyse how metrics from Logistic regression classifier change throughout the layers of `mBERT`.

Before executing, the files `metrics_{language}_per_layer.json` should be uploaded.

In [ ]:
def layersPlot(metric, label=None):
    """
    This function makes a scatter plot of how a metric for a label changes through layers for all languages.
    :param metric: target metric (f1-score or accuracy)
    :param label: target label (S, A or P)
    :returns: 0 when the code is succesfully executed
    """

    fig = go.Figure()

    languages = [
        "Arabic",
        "Basque",
        "English",
        "German",
        "Hindi",
        "Russian",
        "Turkish",
        "Welsh",
    ]

    colors = [
        "#9c179e",
        "#0d0887",
        "#fca636",
        "#bd3786",
        "#46039f",
        "#ed7953",
        "#7201a8",
        "#d8576b",
    ]

    for i in range(len(languages)):
        with open(f"metrics_{languages[i]}_per_layer.json", "r") as file:
            metrics = json.load(file)

        y = [metrics[layer][label][metric] for layer in metrics]

        fig.add_trace(
            go.Scatter(
                x=[i for i in range(13)],
                y=y,
                mode="markers",
                name=languages[i],
                marker=dict(color=colors[i], size=9, opacity=0.9),
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=[i for i in range(13)],
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Layers",
        yaxis_title=f"{metric}",
        margin=dict(l=10, r=10, t=20, b=10),
        width=1000,
        height=600,
    )

    fig.show()
    # pio.write_image(
    #     fig, f"per_layer_{metric}_{label}.png", width=1000, height=600, scale=2
    # )

    return 0

In [ ]:
layersPlot("f1-score", "S")
layersPlot("f1-score", "A")
layersPlot("f1-score", "P")

In [ ]:
def layersPlotSeparate(metric, language):
    """
    This function makes a scatter plot of how a metric for all labels changes through layers for a specific language.
    :param metric: target metric (f1-score or accuracy)
    :param language: target language
    :returns: 0 when the code is succesfully executed
    """

    fig = go.Figure()

    colors = [
        "#9c179e",
        "#0d0887",
        "#d8576b",
    ]

    labels = ["S", "A", "P"]

    with open(f"metrics_{language}_per_layer.json", "r") as file:
        metrics = json.load(file)

    for i in range(len(labels)):
        y = [metrics[layer][labels[i]][metric] for layer in metrics]

        fig.add_trace(
            go.Scatter(
                x=[i for i in range(13)],
                y=y,
                mode="markers",
                name=labels[i],
                marker=dict(color=colors[i], size=9, opacity=0.9),
            )
        )

    fig.update_xaxes(
        tickmode="array",
        tickvals=[i for i in range(13)],
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Layers",
        yaxis_title=f"{metric}",
        margin=dict(l=10, r=10, t=20, b=10),
        width=1000,
        height=600,
    )

    fig.show()
    pio.write_image(
        fig, f"{language}_per_layer_{metric}.png", width=1000, height=600, scale=2
    )

    return 0

In [ ]:
languages = [
    "Arabic",
    "Basque",
    "English",
    "German",
    "Hindi",
    "Russian",
    "Turkish",
    "Welsh",
]

for language in languages:
    layersPlotSeparate("f1-score", language)

### Cross-lingual evaluation: subjecthood across the languages

This subsection is dedicated to testing all fine-tuned `mBERT` models on all languages from the sample. Overall, there are $64$ train-test pairs.

In [ ]:
testPaths = {
    "/content/data/Arabic/ar_padt-ud-test.conllu": "Arabic",
    "/content/data/Basque/eu_bdt-ud-test.conllu": "Basque",
    "/content/data/English/en_ewt-ud-test.conllu": "English",
    "/content/data/German/de_gsd-ud-test.conllu": "German",
    "/content/data/Hindi/hi_hdtb-ud-test.conllu": "Hindi",
    "/content/data/Russian/ru_gsd-ud-test.conllu": "Russian",
    "/content/data/Turkish/tr_penn-ud-test.conllu": "Turkish",
    "/content/data/Welsh/cy_ccg-ud-test.conllu": "Welsh",
}

In [ ]:
for language in tqdm.tqdm(data):
    tokenizer = AutoTokenizer.from_pretrained(data[language]["model"])
    for testPath in testPaths:
        source = language
        target = testPaths[testPath]

        metrics = evaluate(data[language]["model"], testPath)
        with open(f"/content/data/metrics/metrics_{source}_{target}.json", "w") as file:
            json.dump(metrics, file, indent=4)

In [ ]:
def crosslingPlot(metric, label="None"):
    """
    This function makes a heatmap for a specific metric and label.
    :param metric: target metric (f1-score or accuracy)
    :param label: target label (S, A or P)
    :returns: 0 when the code is successfully executed
    """

    languages = [
        "Arabic",
        "Basque",
        "English",
        "German",
        "Hindi",
        "Russian",
        "Turkish",
        "Welsh",
    ]

    matrix = [[None for j in range(len(languages))] for i in range(len(languages))]

    text = [["" for j in range(len(languages))] for i in range(len(languages))]

    for i in range(len(languages)):
        for j in range(len(languages)):
            with open(f"metrics_{languages[i]}_{languages[j]}.json", "r") as file:
                metrics = json.load(file)
                if metric != "accuracy":
                    matrix[i][j] = round(
                        metrics["eval_classification_report"][label][metric], 3
                    )
                else:
                    matrix[i][j] = round(metrics["eval_accuracy"], 3)
                text[i][j] = str(matrix[i][j])

    fig = go.Figure(
        data=go.Heatmap(
            z=matrix,
            x=languages,
            y=languages,
            colorscale="Plasma",
            text=text,
            texttemplate="%{text}",
            textfont={"size": 14},
            showscale=True,
        )
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        font_size=14,
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Test",
        yaxis_title="Train",
        width=700,
        height=700,
        margin=dict(l=10, r=10, t=20, b=10),
    )

    fig.show()
    # pio.write_image(
    #     fig, f"Heatmap_{metric}_{label}.png", width=700, height=700, scale=2
    # )

    return 0

In [ ]:
crosslingPlot("f1-score", "A")
crosslingPlot("f1-score", "P")
crosslingPlot("f1-score", "S")
crosslingPlot("accuracy")

### Analyzing the non-dominant word order

This subsection is dedicated to testing all fine-tuned `mBERT` models on data with a non-dominant word order.

#### Experimental setup

Here, I provide functions that are essential for excluding dominant word order from the data and evaluating `mBERT`.

In [ ]:
def evaluateWordOder(language, testPath, filter, filterType, threshold=None):
    """
    This function exclude dominant word order from the data, evaluates the model and reports the metrics and logs the mistakes.
    :param language: the target language
    :param testPath: the path to the test data
    :param filter: list of the strings that represent the word order, used for filtration of the data
    :filterType: `include` in or `exclude` from the data the word orders from filter
    :returns: confusion matrix
    """

    testDataset = Dataset.from_dict(preprocessData(testPath, threshold, filter, filterType))

    testTokenized = testDataset.map(
        tokenize, batched=True, batch_size=None, remove_columns=testDataset.column_names
    )

    model = AutoModelForTokenClassification.from_pretrained(
        modelId, num_labels=4, device_map="cuda"
    )

    model.eval()

    testTokenized.set_format(
        type="torch", columns=["input_ids", "attention_mask", "labels"]
    )

    testDataLoader = DataLoader(testTokenized, batch_size=8)

    label_list = ["None", "S", "A", "P"]
    mistakes_matrix = [[0 for i in range(4)] for i in range(4)]

    true_predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm.tqdm(testDataLoader):
            batch = {k: v.to("cuda") for k, v in batch.items()}

            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]

            output = model(input_ids=input_ids, attention_mask=attention_mask)

            labels = batch["labels"]
            predictions = torch.argmax(output.logits, dim=-1)

            for prediction, label in zip(predictions, labels):
                true_labels.append(
                    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
                )
                true_predictions.append(
                    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                )

    with open(f"{language}_logs_{filterType}_{'-'.join(filter)}.tsv", "w") as file:
        file.write(f"language\tsentence id\tsentence length\tset of arguments\ttoken id\ttoken\treal label\tpredicted label\n")
        for sent, pred, label in zip(testDataset, true_predictions, true_labels):
            for i in range(len(pred)):
                if pred[i] != label[i]:
                    file.write(f"{language}\t{sent["id"]}\t{len(pred)}\t{' '.join([x for x in label if x != "None"])}\t{i + 1}\t{sent["tokens"][i]}\t{label[i]}\t{pred[i]}\n")

                    x = label_list.index(pred[i])
                    y = label_list.index(label[i])

                    mistakes_matrix[x][y] += 1

    return mistakes_matrix

#### Results

In this subsection, I evaluate `mBERT` on data with the non-dominant word order.

In [ ]:
def confusionMatrixVisualization(language, mistakes):
    """
    This function plots a heatmap based on confusion matrix (the result of conducted experiments that analyze the word order).
    :argument language: the target language
    :argument mistakes: the confusion matrix
    :returns: 0 when code is succesfully executed
    """
    label_list = ["None", "S", "A", "P"]
    text = [[str(mistakes[i][j]) for j in range(len(mistakes))] for i in range(len(mistakes))]

    fig = go.Figure(
        data=go.Heatmap(
            z=mistakes,
            x=label_list,
            y=label_list,
            colorscale="Plasma",
            text=text,
            texttemplate="%{text}",
            textfont={"size": 14},
            showscale=True,
        )
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        font_size=14,
        title_font_family="Brill",
        xaxis_title="Real Label",
        yaxis_title="Predicted Label",
        template="plotly_white",
        width=700,
        height=700,
        margin=dict(l=10, r=10, t=20, b=10),
    )

    fig.show()
    # pio.write_image(
    #     fig, f"Heatmap_mistakes_{language}.png", width=700, height=700, scale=2
    # )

    return 0

**Arabic**
- `VSO` and `VS` are excluded

In [ ]:
testPath = "/content/data/Arabic/ar_padt-ud-test.conllu"
modelId = data["Arabic"]["model"]

language = "Arabic"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["VSO", "VS"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**Basque**
- `SOV` and `SV` are excluded

In [ ]:
testPath = "/content/data/Basque/eu_bdt-ud-test.conllu"
modelId = data["Basque"]["model"]

language = "Basque"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SOV", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**English**
- `SVO` and `SV` are excluded

In [ ]:
testPath = "/content/data/English/en_ewt-ud-test.conllu"
modelId = data["English"]["model"]

language = "English"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SVO", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**German**
- `SVO`, `SOV` and `SV` are excluded

In [ ]:
testPath = "/content/data/German/de_gsd-ud-test.conllu"
modelId = data["German"]["model"]

language = "German"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SVO", "SOV", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**Hindi**
- `SOV` and `SV` are excluded

In [ ]:
testPath = "/content/data/Hindi/hi_hdtb-ud-test.conllu"
modelId = data["Hindi"]["model"]

language = "Hindi"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SOV", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**Russian**
- `SOV` and `SV` are excluded

In [ ]:
testPath = "/content/data/Russian/ru_gsd-ud-test.conllu"
modelId = data["Russian"]["model"]

language = "Russian"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SVO", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**Turkish**
- `SOV` and `SV` are excluded

In [ ]:
testPath = "/content/data/Turkish/tr_penn-ud-test.conllu"
modelId = data["Turkish"]["model"]

language = "Turkish"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["SOV", "SV"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

**Welsh**
- `VSO` and `VS` are excluded

In [ ]:
testPath = "/content/data/Welsh/cy_ccg-ud-test.conllu"
modelId = data["Welsh"]["model"]

language = "Welsh"

tokenizer = AutoTokenizer.from_pretrained(modelId)

filter = ["VSO", "VS"]
filterType = "exclude"

mistakes = evaluateWordOder(language, testPath, filter, filterType)

confusionMatrixVisualization(language, mistakes)

### Extracting Embeddings of `S`, `A` and `P`

This subsection is dedicated to extracting the embeddings of `S`, `A` and `P` arguments from different layers of the fine-tuned `mBERT`. Then, they are visualized with PCA.

#### Experimental setup

In [ ]:
def extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer):
    """
    This function extracts contextual embeddings from a model of the target arguments from the target layers.
    :path: path to the text data in conllu format
    :language: the language of the data
    :modelName: the path to the fine-tuned model from HF
    :targetRole: the target argument whose embedding needs to be extracted
    :pos: the POS of the target argument whose embedding needs to be extracted
    :returns: 0 when code is succesfully executed
    """
    tokenizer = AutoTokenizer.from_pretrained(modelName)

    model = AutoModel.from_pretrained(modelName, output_hidden_states=True)
    model.to("cuda")
    model.eval()

    dataset = Dataset.from_dict(
        preprocessData(path, threshold=None, filter=[], filterType=None)
    )

    embeddings = []

    for i in tqdm.tqdm(range(len(dataset))):
        tokens = dataset[i]["tokens"]
        labels = dataset[i]["label"]

        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            padding=True,
        )
        word_ids = inputs.word_ids()
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model(**inputs)
            hiddenState = output.hidden_states[targetLayer].squeeze(0)

        wordEmbeddings = {}

        for idx, word_idx in enumerate(word_ids):
            if word_idx is not None:
                if labels[word_idx] == targetRole:
                    if word_idx not in wordEmbeddings:
                        wordEmbeddings[word_idx] = []
                    wordEmbeddings[word_idx].append(hiddenState[idx])

        for token in wordEmbeddings:
            embeddings.append(torch.stack(wordEmbeddings[token]).mean(0).cpu().numpy())

    torch.save(
        embeddings, f"{language}_{targetRole}_{pos}_layer{targetLayer}_embeddings.pt"
    )

    return 0

#### Results

**Arabic**

In [ ]:
language = "Arabic"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**Basque**

In [ ]:
language = "Basque"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**English**

In [ ]:
language = "English"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**German**

In [ ]:
language = "German"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**Hindi**

In [ ]:
language = "Hindi"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**Russian**

In [ ]:
language = "Russian"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [0, 4, 8, -1]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**Turkish**

In [ ]:
language = "Turkish"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

**Welsh**

In [ ]:
language = "Welsh"
path = data[language]["path"] + "-train.conllu"
modelName = data[language]["model"]
pos = "NOUN"  # pos = "PRON"
for targetRole in [1, 2, 3]:
    for targetLayer in [-1, 0, 4, 8]:
        extractEmbeddings(path, language, modelName, targetRole, pos, targetLayer)

#### Visualization

In [ ]:
pathDict = {}

prefixPath = "/content/embeddings/"

dirToPos = {"nouns": "NOUN", "pronouns": "PRON"}
dirToLayer = {
    "layer 0": "layer0",
    "layer 4": "layer4",
    "layer 8": "layer8",
    "layer 12": "layer-1",
}

for language in [
    "Arabic",
    "Basque",
    "English",
    "German",
    "Hindi",
    "Russian",
    "Turkish",
    "Welsh",
]:
    pathDict[language] = {}
    for layer in ["layer 0", "layer 4", "layer 8", "layer 12"]:
        pathDict[language][layer] = {}
        for role in ["1", "2", "3"]:
            pathDict[language][layer][role] = {}

            pathDict[language][layer][role]["noun"] = (
                prefixPath
                + f"nouns/{layer}/{language}_{role}_NOUN_{dirToLayer[layer]}_embeddings.pt"
            )

            pathDict[language][layer][role]["pronoun"] = (
                prefixPath
                + f"pronouns/{layer}/{language}_{role}_PRON_{dirToLayer[layer]}_embeddings.pt"
            )

In [ ]:
colors = {"1": "#7201a8", "2": "#fca636", "3": "#d8576b"}
id2arg = {"1": "S", "2": "A", "3": "P"}
layer2id = {"layer 0": 1, "layer 4": 2, "layer 8": 3, "layer 12": 4}

for language in tqdm.tqdm(pathDict):
    fig = make_subplots(
        rows=1, cols=4, subplot_titles=["Layer 0", "Layer 4", "Layer 8", "Classifier"]
    )
    for layer in tqdm.tqdm(pathDict[language]):
        embS = np.vstack(
            (
                torch.load(
                    pathDict[language][layer]["1"]["noun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
                torch.load(
                    pathDict[language][layer]["1"]["pronoun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
            )
        )

        embA = np.vstack(
            (
                torch.load(
                    pathDict[language][layer]["2"]["noun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
                torch.load(
                    pathDict[language][layer]["2"]["pronoun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
            )
        )

        embP = np.vstack(
            (
                torch.load(
                    pathDict[language][layer]["3"]["noun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
                torch.load(
                    pathDict[language][layer]["3"]["pronoun"],
                    map_location=torch.device("cpu"),
                    weights_only=False,
                ),
            )
        )

        labels = np.array(["1"] * len(embS) + ["2"] * len(embA) + ["3"] * len(embP))

        pca = PCA(n_components=2)

        embeddings = pca.fit_transform(np.vstack((embS, embA, embP)))

        for role in ["1", "2", "3"]:
            idx = labels == role

            fig.add_trace(
                go.Scatter(
                    x=embeddings[idx, 0],
                    y=embeddings[idx, 1],
                    mode="markers",
                    name=id2arg[role],
                    marker=dict(color=colors[role], opacity=0.7),
                ),
                row=1,
                col=layer2id[layer],
            )

        fig.update_xaxes(title_text="x", row=1, col=layer2id[layer])
        fig.update_yaxes(title_text="y", row=1, col=layer2id[layer])

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        title_font_family="Brill",
        template="plotly_white",
        margin=dict(l=10, t=25, b=20, r=10),
        showlegend=False,
    )
    fig.show()
    # pio.write_image(
    #     fig, f"{language}_embeddings_per_layers_.png", width=2000, height=600, scale=2
    # )

In [ ]:
colors = {
    "Arabic": "#9c179e",
    "Basque": "#0d0887",
    "English": "#fca636",
    "German": "#bd3786",
    "Hindi": "#46039f",
    "Russian": "#ed7953",
    "Turkish": "#7201a8",
    "Welsh": "#d8576b",
}

layer = "layer 12"

fig = make_subplots(rows=1, cols=3, subplot_titles=["S", "A", "P"])

for role in tqdm.tqdm(["1", "2", "3"]):
    embeddings = []
    labels = []

    for language in pathDict:
        nouns = torch.load(
            pathDict[language][layer][role]["noun"],
            map_location=torch.device("cpu"),
            weights_only=False,
        )

        pronouns = torch.load(
            pathDict[language][layer][role]["pronoun"],
            map_location=torch.device("cpu"),
            weights_only=False,
        )

        embeddings.append(nouns)
        embeddings.append(pronouns)

        labels.append([language] * (len(nouns) + len(pronouns)))

    pca = PCA(n_components=2)

    labels = np.hstack(labels)
    embeddings = pca.fit_transform(np.vstack(embeddings))

    for language in pathDict:
        idx = labels == language

        fig.add_trace(
            go.Scatter(
                x=embeddings[idx, 0],
                y=embeddings[idx, 1],
                mode="markers",
                name=language,
                marker=dict(color=colors[language], opacity=0.7),
                showlegend=True if role == "3" else False,
            ),
            row=1,
            col=int(role),
        )

    fig.update_xaxes(title_text="x", row=1, col=int(role))
    fig.update_yaxes(title_text="y", row=1, col=int(role))

fig.update_layout(
    font_family="Brill",
    font_color="black",
    title_font_family="Brill",
    template="plotly_white",
    margin=dict(l=10, t=25, b=20, r=10),
)
fig.show()
# pio.write_image(fig, f"crossling_embeddings.png", width=2000, height=600, scale=2)

In [ ]:
colors = {"noun": "#9c179e", "pronoun": "#fca636"}

markers = {"noun": "x", "pronoun": "triangle-up"}

layer = "layer 12"

for language in tqdm.tqdm(pathDict):
    fig = make_subplots(rows=1, cols=3, subplot_titles=["S", "A", "P"])
    for role in tqdm.tqdm(["1", "2", "3"]):
        nouns = torch.load(
            pathDict[language][layer][role]["noun"],
            map_location=torch.device("cpu"),
            weights_only=False,
        )

        pronouns = torch.load(
            pathDict[language][layer][role]["pronoun"],
            map_location=torch.device("cpu"),
            weights_only=False,
        )

        embeddings = np.vstack((nouns, pronouns))

        labels = np.array(["Noun"] * len(nouns) + ["Pronoun"] * len(pronouns))

        pca = PCA(n_components=2)

        embeddings = pca.fit_transform(np.vstack(embeddings))

        for pos in ["Noun", "Pronoun"]:
            idx = labels == pos

            fig.add_trace(
                go.Scatter(
                    x=embeddings[idx, 0],
                    y=embeddings[idx, 1],
                    mode="markers",
                    name=pos,
                    marker=dict(
                        symbol=markers[pos.lower()],
                        color=colors[pos.lower()],
                        opacity=0.7,
                    ),
                    showlegend=True if role == "3" else False,
                ),
                row=1,
                col=int(role),
            )
        cnt += 1

        fig.update_xaxes(title_text="x", row=1, col=int(role))
        fig.update_yaxes(title_text="y", row=1, col=int(role))

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        title_font_family="Brill",
        template="plotly_white",
        margin=dict(l=10, t=25, b=20, r=10),
    )
    fig.show()
    # pio.write_image(
    #     fig,
    #     f"{language}_noun_pronoun_crossling_embeddings.png",
    #     width=1000,
    #     height=500,
    #     scale=2,
    # )

### Analyzing attention distribution

This subsection is dedicated to analyzing the shift of attention throughout all layers and heads before and after fine-tuning `mBERT`.

#### Experimental setup

In [ ]:
def extractAttention(path, language, modelName, state):
    """
    This function extracts attention from the target model and calculates the mean entropy throughout all layers and heads on target data.
    :path: path to the text data in conllu format
    :language: the language of the data
    :modelName: the path to the target (base or fine-tuned) model from HF
    :state: the state of the model (base / fine-tuned)
    :returns: 0 when code is succesfully executed
    """
    tokenizer = AutoTokenizer.from_pretrained(modelName)

    model = AutoModel.from_pretrained(modelName, output_attentions=True)
    model.to("cuda")
    model.eval()

    dataset = Dataset.from_dict(
        preprocessData(path, threshold=None, filter=[], filterType=None)
    )

    attentionMatrix = torch.zeros(12, 12)

    for i in range(len(dataset)):
        tokens = dataset[i]["tokens"]

        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            padding=True,
        )

        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model(**inputs, output_attentions=True)

        attention = output.attentions

        for layer in range(12):
            attentionPerLayer = attention[layer].squeeze()
            for head in range(12):
                p = attentionPerLayer[head][0]
                attentionMatrix[layer][head] += (
                    -(p * torch.log(p + 1e-9)).sum(dim=-1).cpu().numpy()
                )

    for layer in range(12):
        for head in range(12):
            attentionMatrix[layer][head] /= len(dataset)

    torch.save(attentionMatrix, f"{language}_{state}_matrix.pt")

    return 0

#### Results

In [ ]:
for language in tqdm.tqdm(data):
    baseModel = "bert-base-multilingual-cased"
    finetunedModel = data[language]["model"]

    extractAttention(
        data[language]["path"] + "-test.conllu", language, baseModel, "base"
    )

    extractAttention(
        data[language]["path"] + "-test.conllu", language, finetunedModel, "fine-tuned"
    )

#### Visualization

The visualization of attention distribution before fine-tuning.

In [ ]:
for language in tqdm.tqdm(data):
    base = torch.load(
        f"/content/data/{language}_base_matrix.pt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    text = [
        [str(round(base[i][j].item(), 2)) for j in range(len(base))]
        for i in range(len(base))
    ]

    fig = go.Figure(
        data=go.Heatmap(
            z=base,
            x=[i for i in range(1, 13)],
            y=[i for i in range(1, 13)],
            colorscale="Plasma",
            text=text,
            texttemplate="%{text}",
            textfont={"size": 14},
            showscale=True,
        )
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        font_size=14,
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Heads",
        yaxis_title="Layers",
        width=700,
        height=700,
        margin=dict(l=10, r=10, t=20, b=10),
    )

    fig.show()
    # pio.write_image(
    #     fig, f"Heatmap_{language}_base_entropy.png", width=700, height=700, scale=2
    # )

The visualization of attention distribution after fine-tuning.

In [ ]:
for language in tqdm.tqdm(data):
    finetuned = torch.load(
        f"/content/data/{language}_fine-tuned_matrix.pt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    text = [
        [str(round(finetuned[i][j].item(), 2)) for j in range(len(finetuned))]
        for i in range(len(finetuned))
    ]

    fig = go.Figure(
        data=go.Heatmap(
            z=finetuned,
            x=[i for i in range(1, 13)],
            y=[i for i in range(1, 13)],
            colorscale="Plasma",
            text=text,
            texttemplate="%{text}",
            textfont={"size": 14},
            showscale=True,
        )
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        font_size=14,
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Heads",
        yaxis_title="Layers",
        width=700,
        height=700,
        margin=dict(l=10, r=10, t=20, b=10),
    )
    fig.show()
    # pio.write_image(
    #     fig, f"Heatmap_{language}_finetuned_entropy.png", width=700, height=700, scale=2
    # )

The visualization of delta in attention distribution.

In [ ]:
for language in tqdm.tqdm(data):
    base = torch.load(
        f"/content/data/{language}_base_matrix.pt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    finetuned = torch.load(
        f"/content/data/{language}_fine-tuned_matrix.pt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    delta = finetuned - base

    text = [
        [str(round(delta[i][j].item(), 2)) for j in range(len(delta))]
        for i in range(len(delta))
    ]

    fig = go.Figure(
        data=go.Heatmap(
            z=finetuned - base,
            x=[i for i in range(1, 13)],
            y=[i for i in range(1, 13)],
            text=text,
            texttemplate="%{text}",
            textfont={"size": 14},
            colorscale="Plasma",
            showscale=True,
        )
    )

    fig.update_layout(
        font_family="Brill",
        font_color="black",
        font_size=14,
        title_font_family="Brill",
        template="plotly_white",
        xaxis_title="Heads",
        yaxis_title="Layers",
        width=700,
        height=700,
        margin=dict(l=10, r=10, t=20, b=10),
    )
    fig.show()
    # pio.write_image(
    #     fig, f"Heatmap_{language}_delta_entropy.png", width=700, height=700, scale=2
    # )